# 02 — Random Forest Classifier

Train Random Forest on eNose sensor data aligned to MetaBreath feature space.

**Inputs**: `data/processed/X_train.csv`, `y_train.csv`, `X_test.csv`, `y_test.csv`
**Output**: `apps/api/models/rf_classifier.joblib`

In [ ]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

ROOT = Path('../../..')
PROCESSED = ROOT / 'data' / 'processed'
MODEL_DIR = ROOT / 'apps' / 'api' / 'models'
MODEL_DIR.mkdir(exist_ok=True)

X_train = pd.read_csv(PROCESSED / 'X_train.csv')
X_test  = pd.read_csv(PROCESSED / 'X_test.csv')
y_train = pd.read_csv(PROCESSED / 'y_train.csv').squeeze()
y_test  = pd.read_csv(PROCESSED / 'y_test.csv').squeeze()

print('Train:', X_train.shape, '| Test:', X_test.shape)

## Train Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, f1_score
)
import matplotlib.pyplot as plt
import seaborn as sns

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)

## Evaluation — F1, Confusion Matrix, ROC-AUC

In [ ]:
y_pred = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]

print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['Normal', 'Diabetes']))

f1 = f1_score(y_test, y_pred, average='weighted')
roc = roc_auc_score(y_test, y_proba)
print(f'Weighted F1: {f1:.4f}')
print(f'ROC-AUC:     {roc:.4f}')

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Diabetes'],
            yticklabels=['Normal', 'Diabetes'], ax=ax)
ax.set_title(f'Random Forest — F1={f1:.3f}, AUC={roc:.3f}')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
plt.tight_layout()
plt.savefig(MODEL_DIR / 'rf_confusion_matrix.png', dpi=150)
plt.show()

## Feature Importance

In [ ]:
importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=importance_df, x='importance', y='feature', ax=ax)
ax.set_title('RF Feature Importance')
plt.tight_layout()
plt.savefig(MODEL_DIR / 'rf_feature_importance.png', dpi=150)
plt.show()
print(importance_df)

## Save Model

In [ ]:
out_path = MODEL_DIR / 'rf_classifier.joblib'
joblib.dump(rf, out_path)
print(f'Saved: {out_path}')

# Save metrics
import json
metrics = {'f1_weighted': round(f1, 4), 'roc_auc': round(roc, 4), 'n_train': len(X_train), 'n_test': len(X_test)}
with open(MODEL_DIR / 'rf_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics:', metrics)